# Bronze layer ingestion
Reads incrementally arriving CSV files from `raw_landing` using Auto Loader and writes them into Delta tables under `olist.bronze`, with schema inference/evolution and ingestion metadata columns.

In [0]:
LANDING_DIR = "/Volumes/olist/bronze/raw_landing/"
SCHEMA_DIR = "/Volumes/olist/bronze/pipeline_metadata/_schema/"
CHECKPOINT_DIR = "/Volumes/olist/bronze/pipeline_metadata/_checkpoint/"

TABLES = [
    "olist_customers_dataset",
    "olist_orders_dataset",
    "olist_order_items_dataset",
    "olist_order_payments_dataset",
    "olist_order_reviews_dataset",
    "olist_products_dataset",
    "olist_sellers_dataset",
    "olist_geolocation_dataset",
    "product_category_name_translation",
]

## Ingestion function
Each table gets its own Auto Loader stream, filtered by filename prefix, with its own schema location and checkpoint. `trigger(availableNow=True)` processes what's currently available and stops: appropriate for a batch-style source like this one.

In [0]:
from pyspark.sql.functions import current_timestamp, col

def ingest_table(table_name):
    (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", f"{SCHEMA_DIR}{table_name}/")
        .option("header", "true")
        .option("pathGlobFilter", f"{table_name}_*.csv")
        .load(LANDING_DIR)
        .withColumn("_ingest_timestamp", current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
        .writeStream
        .option("checkpointLocation", f"{CHECKPOINT_DIR}{table_name}/")
        .trigger(availableNow=True)
        .toTable(f"olist.bronze.{table_name}")
    )

In [0]:
for table in TABLES:
    print(f"Ingesting {table}...")
    ingest_table(table)
    print(f"Done: olist.bronze.{table}")

In [0]:
%sql
SHOW TABLES IN olist.bronze;

In [0]:
%sql
-- Quick row-count check on 2 tables as a sample.
-- Extend with UNION ALL for the remaining 7 tables if a full check is needed.
SELECT 'olist_orders_dataset' AS table_name, COUNT(*) AS row_count FROM olist.bronze.olist_orders_dataset
UNION ALL
SELECT 'olist_customers_dataset', COUNT(*) FROM olist.bronze.olist_customers_dataset;